# 🦵 RSNA Abnormality Detection — EfficientNet 2.5D Baseline

> **Competition:** [RSNA Knee Abnormality Detection](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection)  
> **Strategy:** 2.5D EfficientNetV2-S on middle slices of each series, study-level attention pooling, 12-label sigmoid head  
> **Metric:** Macro-averaged AUC-ROC  

### Architecture at a glance
```
For each MRI series:
  • Sample N_SLICES middle slices (2.5D: stack 3 adjacent → 3-channel "RGB")
  • EfficientNetV2-S backbone → 1280-d embedding per slice group
  • Average pool over slices → series embedding
For each study:
  • Weighted average over series embeddings (learnable weights)
  • 12-head linear → sigmoid → 12 probabilities
Loss: weighted binary cross-entropy per label
```


In [ ]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm

warnings.filterwarnings('ignore')

# ── Config ────────────────────────────────────────────────────────────────────
class CFG:
    seed          = 42
    debug         = False          # set True to run 1 batch per epoch for speed
    base          = Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')
    model_name    = 'efficientnetv2_s'
    batch_size    = 2      # was 8
    img_size      = 224    # was 256
    n_slices      = 3      # was 5
    in_chans      = 3              # adjacent-slice stacking (2.5D)
    epochs        = 8
    lr            = 2e-4
    wd            = 1e-4
    fold          = 0
    n_folds       = 3
    amp           = True
    device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TARGETS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus',
           'Medial OA','Lateral OA','PF OA','Effusion',
           'Synovitis',"Baker's",'Contusion','Fracture']

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)
print(f'Device: {CFG.device}')
print(f'AMP: {CFG.amp}')

In [ ]:
# ── Data Loading ──────────────────────────────────────────────────────────────
train = pd.read_csv(CFG.base / 'train.csv')
train_series = pd.read_csv(CFG.base / 'train_series.csv')
test  = pd.read_csv(CFG.base / 'test.csv')
test_series  = pd.read_csv(CFG.base / 'test_series.csv')

# Only use studies with labels
labeled_mask = train[TARGETS].notna().all(axis=1)
train_lab = train[labeled_mask].reset_index(drop=True)
print(f'Labeled studies: {len(train_lab):,}')

# Merge series info
train_merged = train_lab.merge(train_series, on='StudyInstanceUID', how='left')
test_merged  = test.merge(test_series, on='StudyInstanceUID', how='left')

# Build DICOM file list
def get_dcm_paths(base_dir, study_id, series_id):
    folder = Path(base_dir) / study_id / series_id
    return sorted(folder.glob('*.dcm'))

print(train_merged.head(3))

In [ ]:
# ── Cross-validation split ────────────────────────────────────────────────────
# Stratify on most common label (Medial Meniscus tends to be most frequent)
sgkf = StratifiedGroupKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train_lab['fold'] = -1
for fold, (_, val_idx) in enumerate(
    sgkf.split(train_lab, train_lab['ACL'].fillna(0), groups=train_lab['StudyInstanceUID'])
):
    train_lab.loc[val_idx, 'fold'] = fold

print(train_lab['fold'].value_counts().sort_index())

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────
try:
    import pydicom
    from PIL import Image
    import torchvision.transforms as T
except ImportError as e:
    raise RuntimeError(f'Missing dependency: {e}. Add pydicom and Pillow.')

def dcm_to_array(path):
    """Load a DICOM slice and return a normalised float32 array [H,W]."""
    ds = pydicom.dcmread(str(path))
    img = ds.pixel_array.astype(np.float32)
    # Apply window-center / window-width if available
    wc = getattr(ds, 'WindowCenter', None)
    ww = getattr(ds, 'WindowWidth', None)
    if wc is not None and ww is not None:
        wc = float(wc[0]) if hasattr(wc, '__len__') else float(wc)
        ww = float(ww[0]) if hasattr(ww, '__len__') else float(ww)
        lo, hi = wc - ww / 2, wc + ww / 2
        img = np.clip(img, lo, hi)
        img = (img - lo) / (ww + 1e-6)
    else:
        img = (img - img.min()) / (img.max() - img.min() + 1e-6)
    return img


def load_series_slices(dcm_paths, n_slices, in_chans, img_size):
    """
    Load N slices from the middle of a series.
    Stack adjacent slices to form 3-channel pseudo-RGB (2.5D).
    Returns tensor of shape [n_slices, in_chans, H, W].
    """
    n = len(dcm_paths)
    mid = n // 2
    half = n_slices // 2
    indices = list(range(max(0, mid - half), min(n, mid + half + n_slices % 2)))
    # Pad if needed
    while len(indices) < n_slices:
        indices.append(indices[-1])

    arrays = []
    for idx in indices:
        arr = dcm_to_array(dcm_paths[min(idx, n - 1)])
        arr = np.array(Image.fromarray(arr).resize((img_size, img_size), Image.BILINEAR))
        arrays.append(arr)

    # 2.5D stacking: each output channel = average of in_chans adjacent slices
    result = []
    for i in range(n_slices):
        channels = []
        for offset in range(-(in_chans // 2), in_chans // 2 + 1):
            j = max(0, min(len(arrays) - 1, i + offset))
            channels.append(arrays[j])
        result.append(np.stack(channels[:in_chans], axis=0))  # [C, H, W]

    return torch.tensor(np.stack(result, axis=0), dtype=torch.float32)  # [N, C, H, W]


class KneeDataset(Dataset):
    def __init__(self, df, df_series, base_dir, split='train', targets=TARGETS):
        self.df = df.reset_index(drop=True)
        self.df_series = df_series
        self.base_dir = Path(base_dir)
        self.split = split
        self.targets = targets
        self.augment = T.Compose([
            T.RandomHorizontalFlip(p=0.5),
            T.RandomVerticalFlip(p=0.1),
        ]) if split == 'train' else nn.Identity()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        study_id = row['StudyInstanceUID']

        # Get all series for this study
        series_rows = self.df_series[self.df_series['StudyInstanceUID'] == study_id]

        series_tensors = []
        series_meta = []
        for _, srow in series_rows.iterrows():
            sid = srow['SeriesInstanceUID']
            dcm_folder = self.base_dir / study_id / sid
            paths = sorted(dcm_folder.glob('*.dcm'))
            if not paths:
                continue
            try:
                slices = load_series_slices(paths, CFG.n_slices, CFG.in_chans, CFG.img_size)
                series_tensors.append(slices)  # [N_slices, C, H, W]
                series_meta.append([
                    float(srow.get('Fluid_Sensitive', 0)),
                    float(srow.get('Fat_Suppression', 0)),
                    float({'Sagittal': 0, 'Coronal': 1, 'Axial': 2}.get(
                          srow.get('Anatomical_Plane', 'Sagittal'), 0)) / 2.0
                ])
            except Exception:
                continue

        if not series_tensors:
            # Fallback: zeros
            series_tensors = [torch.zeros(CFG.n_slices, CFG.in_chans,
                                          CFG.img_size, CFG.img_size)]
            series_meta = [[0., 0., 0.]]

        # Pad / truncate to max_series = 6 series per study
        MAX_SERIES = 6
        series_tensors = series_tensors[:MAX_SERIES]
        series_meta = series_meta[:MAX_SERIES]
        while len(series_tensors) < MAX_SERIES:
            series_tensors.append(torch.zeros_like(series_tensors[0]))
            series_meta.append([0., 0., 0.])

        x = torch.stack(series_tensors, dim=0)      # [S, N, C, H, W]
        meta = torch.tensor(series_meta, dtype=torch.float32)  # [S, 3]
        mask = torch.zeros(MAX_SERIES, dtype=torch.bool)       # [S]
        mask[:len(series_rows.head(MAX_SERIES))] = True

        if self.split in ('train', 'valid'):  # Return real labels for both train and validation
            labels = torch.tensor(
                row[self.targets].values.astype(np.float32), dtype=torch.float32
            )
        else:
            labels = torch.zeros(len(self.targets))  # Test set gets zeros

        return x, meta, mask, labels, study_id

print('Dataset class defined ✓')

In [ ]:
# ── Model ─────────────────────────────────────────────────────────────────────
class KneeModel(nn.Module):
    def __init__(self, backbone='efficientnetv2_s', n_targets=12, n_series=6, feat_dim=1280):
        super().__init__()
        self.n_series = n_series
        self.backbone = timm.create_model(
            backbone, pretrained=False, in_chans=CFG.in_chans,
            num_classes=0, global_pool='avg'
        )
        feat_dim = self.backbone.num_features

        # Series-level meta encoder
        self.meta_enc = nn.Sequential(
            nn.Linear(3, 32), nn.ReLU(), nn.Linear(32, feat_dim)
        )

        # Study-level attention over series
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, 128), nn.Tanh(), nn.Linear(128, 1)
        )

        # Classification head
        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim, n_targets)
        )

    def forward(self, x, meta, mask):
        # x: [B, S, N, C, H, W]
        B, S, N, C, H, W = x.shape

        # Encode all slices across all series
        x_flat = x.view(B * S * N, C, H, W)          # [B*S*N, C, H, W]
        feats_flat = self.backbone(x_flat)              # [B*S*N, D]
        feats = feats_flat.view(B, S, N, -1)           # [B, S, N, D]
        D = feats.shape[-1]

        # Pool over slices
        series_feats = feats.mean(dim=2)               # [B, S, D]

        # Add meta signal
        meta_feats = self.meta_enc(meta)               # [B, S, D]
        series_feats = series_feats + meta_feats

        # Attention pooling over series (masked)
        attn_logits = self.attn(series_feats).squeeze(-1)  # [B, S]
        attn_logits = attn_logits.masked_fill(~mask, float('-inf'))
        attn_weights = torch.softmax(attn_logits, dim=-1)  # [B, S]
        study_feat = (attn_weights.unsqueeze(-1) * series_feats).sum(dim=1)  # [B, D]

        return self.head(study_feat)  # [B, 12] logits

model = KneeModel()
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params/1e6:.1f}M')

In [ ]:
print([t for t in TARGETS if t in train.columns])

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────
def compute_pos_weights(df, targets):
    """Compute positive class weights for weighted BCE."""
    weights = []
    for t in targets:
        pos = df[t].sum()
        neg = len(df) - pos
        weights.append(neg / (pos + 1e-6))
    return torch.tensor(weights, dtype=torch.float32).to(CFG.device)


def train_epoch(model, loader, optimizer, scaler, pos_weights):
    model.train()
    total_loss = 0.
    for step, (x, meta, mask, labels, _) in enumerate(loader):
        x, meta, mask, labels = (
            x.to(CFG.device), meta.to(CFG.device),
            mask.to(CFG.device), labels.to(CFG.device)
        )
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=CFG.amp):
            logits = model(x, meta, mask)
            loss = F.binary_cross_entropy_with_logits(
                logits, labels, pos_weight=pos_weights
            )
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        if CFG.debug and step == 0:
            break
    return total_loss / (step + 1)


@torch.no_grad()
def valid_epoch(model, loader):
    model.eval()
    all_logits, all_labels = [], []
    for step, (x, meta, mask, labels, _) in enumerate(loader):
        x, meta, mask = x.to(CFG.device), meta.to(CFG.device), mask.to(CFG.device)
        with torch.cuda.amp.autocast(enabled=CFG.amp):
            logits = model(x, meta, mask)
        all_logits.append(logits.float().cpu())
        all_labels.append(labels)
        if CFG.debug and step == 0:
            break
    preds = torch.sigmoid(torch.cat(all_logits)).numpy()
    labs  = torch.cat(all_labels).numpy()
    aucs = []
    for i, t in enumerate(TARGETS):
        y = labs[:, i]
        if y.sum() > 0 and y.sum() < len(y):   # needs both pos AND neg
            aucs.append(roc_auc_score(y, preds[:, i]))
    return (np.mean(aucs) if aucs else 0.0), preds



# Create fold split
trn_idx = train_lab[train_lab['fold'] != CFG.fold].index.tolist()
val_idx = train_lab[train_lab['fold'] == CFG.fold].index.tolist()
trn_df = train_lab.loc[trn_idx].reset_index(drop=True)
val_df = train_lab.loc[val_idx].reset_index(drop=True)

trn_series_dir = CFG.base / 'train_series'
trn_ds = KneeDataset(trn_df, train_series, trn_series_dir, split='train')
val_ds = KneeDataset(val_df, train_series, trn_series_dir, split='valid')

trn_loader = DataLoader(trn_ds, batch_size=CFG.batch_size, shuffle=True,
                        num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False,
                        num_workers=2, pin_memory=True)

model = KneeModel().to(CFG.device)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.wd)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG.epochs, eta_min=CFG.lr * 0.1
)
scaler = torch.cuda.amp.GradScaler(enabled=CFG.amp)
pos_weights = compute_pos_weights(trn_df, TARGETS)

best_auc = 0.
for epoch in range(1, CFG.epochs + 1):
    trn_loss = train_epoch(model, trn_loader, optimizer, scaler, pos_weights)
    val_auc, val_preds = valid_epoch(model, val_loader)
    scheduler.step()
    print(f'Epoch {epoch:02d} | loss={trn_loss:.4f} | val_AUC={val_auc:.4f}')
    if val_auc > best_auc:
        best_auc = val_auc
        torch.save(model.state_dict(), 'best_model.pt')
        best_preds = val_preds
        best_val_df = val_df.copy()

print(f'\nBest fold {CFG.fold} AUC: {best_auc:.4f}')

In [ ]:
print(f"Labeled studies in Train: {len(trn_df)}")
print(f"Labeled studies in Val:   {len(val_df)}")
print("Val positive counts per target:")
print(val_df[TARGETS].sum())


In [ ]:
# ── Inference & Submission ────────────────────────────────────────────────────
test_series_dir = CFG.base / 'test_series'
test_ds = KneeDataset(test, test_series, test_series_dir, split='test')
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False,
                         num_workers=2, pin_memory=True)

model.load_state_dict(torch.load('best_model.pt', map_location=CFG.device))
model.eval()

all_preds, all_ids = [], []
with torch.no_grad():
    for x, meta, mask, _, study_ids in test_loader:
        x, meta, mask = x.to(CFG.device), meta.to(CFG.device), mask.to(CFG.device)
        with torch.cuda.amp.autocast(enabled=CFG.amp):
            logits = model(x, meta, mask)
        preds = torch.sigmoid(logits).float().cpu().numpy()
        all_preds.append(preds)
        all_ids.extend(list(study_ids))

preds_arr = np.vstack(all_preds)
sub = pd.DataFrame(preds_arr, columns=TARGETS)
sub.insert(0, 'StudyInstanceUID', all_ids)

# Fill missing test studies with 0.5
sub_template = pd.read_csv(CFG.base / 'sample_submission.csv')
sub = sub_template[['StudyInstanceUID']].merge(sub, on='StudyInstanceUID', how='left')
for col in TARGETS:
    sub[col] = sub[col].fillna(0.5)

sub.to_csv('submission.csv', index=False)
print(f'submission.csv shape: {sub.shape}')
sub.head()

## Next Steps to Improve

| Idea | Expected Gain |
|------|---------------|
| **Report NLP** — extract pseudo-labels from the radiology text (mDeBERTa / BiomedBERT) | +0.02–0.05 AUC |
| **Full 3D volumes** — replace 2.5D with 3D UNet / SlowFast | +0.01–0.03 AUC |
| **Larger backbone** — ConvNeXt-XL, ViT-L, or EVA-02 | +0.01–0.02 AUC |
| **TTA** — horizontal flip, multi-scale | +0.005 AUC |
| **Multi-fold ensemble** — average 5 folds | +0.005–0.01 AUC |
| **Plane-aware heads** — separate linear head per anatomical plane | +0.005 AUC |
